# Análise de Sentimento com NLTK

Este notebook demonstra dois exemplos de análise de sentimento usando as ferramentas do NLTK:

1. **Parte 1 — Corpus Subjectivity**: classifica frases de críticas de cinema como *subjetivas* ou *objetivas* usando Naive Bayes.
2. **Parte 2 — Dataset IMDB**: classifica reviews completas como *positive* ou *negative*, também com Naive Bayes.
3. **Parte 3 — VADER**: aplica o analisador léxico VADER (sem treinamento) nos mesmos dados e compara os resultados.

---

## Configuração

1. Instale o NLTK usando pip: `pip install nltk`. O pacote está disponível no arquivo `requirements.txt` para facilitar a instalação.
2. Importe o NLTK e baixe os recursos necessários (célula de código abaixo): `nltk.download()`
3. Na janela de download do NLTK, selecione os recursos `subjectivity` e `vader_lexicon`. Você pode usar a barra de pesquisa para encontrar esses recursos rapidamente.
4. Importe as classes e funções necessárias para análise de sentimento (células abaixo).

In [53]:
# import nltk
# nltk.download()

### Importações

- `NaiveBayesClassifier` — o algoritmo de classificação
- `subjectivity` — o corpus de frases rotuladas
- `SentimentAnalyzer` — orquestrador do pipeline: extração de features, treino e avaliação
- `nltk.sentiment.util.*` — utilitários como `mark_negation` e `extract_unigram_feats`

In [54]:
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *
import pandas as pd

---

## Parte 1: Corpus Subjectivity — Classificação de Subjetividade

O corpus [`subjectivity`](https://www.cs.cornell.edu/people/pabo/movie-review-data/) (Pang & Lee, 2004) contém frases extraídas de resenhas de filmes, rotuladas como:

- **`subj`** — frases *subjetivas*: opiniões, sentimentos e julgamentos pessoais (ex.: *"This film is a masterpiece."*)
- **`obj`** — frases *objetivas*: descrições factuais sem carga emocional (ex.: *"The film stars Tom Hanks."*)

O objetivo desta parte é treinar um classificador Naive Bayes capaz de distinguir automaticamente entre os dois tipos.

### Carregando o corpus

Usamos apenas **100 frases de cada categoria** (de um total de ~5.000) para manter o tempo de execução baixo. Cada documento é representado como uma tupla `(lista_de_tokens, rótulo)`.

In [55]:
n_instances = 100
subj_docs = [(sent, 'subj') for sent in subjectivity.sents(categories='subj')[:n_instances]]
obj_docs = [(sent, 'obj') for sent in subjectivity.sents(categories='obj')[:n_instances]]
len(subj_docs), len(obj_docs)

(100, 100)

In [56]:
subj_docs[0]

(['smart',
  'and',
  'alert',
  ',',
  'thirteen',
  'conversations',
  'about',
  'one',
  'thing',
  'is',
  'a',
  'small',
  'gem',
  '.'],
 'subj')

### Divisão treino / teste

Separamos **80 %** para treino e **20 %** para teste, mantendo o balanceamento entre subjetivas e objetivas (80 + 80 treino, 20 + 20 teste).

In [57]:
train_subj_docs = subj_docs[:80]
test_subj_docs = subj_docs[80:100]
train_obj_docs = obj_docs[:80]
test_obj_docs = obj_docs[80:100]
train_docs = train_subj_docs + train_obj_docs
test_docs = test_subj_docs + test_obj_docs

### Extração de features: negação e unigramas

Antes de treinar, é preciso definir quais características representam cada documento. Usamos dois passos:

1. **`mark_negation`** — percorre cada frase e adiciona o sufixo `_NEG` em todos os tokens que aparecem após uma palavra de negação (ex.: *not*, *never*) até o final da cláusula. Isso permite ao modelo diferenciar `good` de `not good_NEG`.

2. **`unigram_word_feats`** com `min_freq=4` — seleciona apenas os tokens que aparecem ao menos 4 vezes no corpus de treino. Tokens muito raros tendem a ser ruído e prejudicam a generalização.

A célula abaixo mostra quantas features foram selecionadas.

In [58]:
sentim_analyzer = SentimentAnalyzer()
all_words_neg = sentim_analyzer.all_words([mark_negation(doc) for doc in train_docs])

In [59]:
unigram_feats = sentim_analyzer.unigram_word_feats(all_words_neg, min_freq=4)
len(unigram_feats)

83

Com as features definidas, registramos o extrator no `SentimentAnalyzer` e aplicamos sobre treino e teste, transformando cada lista de tokens em um dicionário `{feature: True/False}` pronto para o classificador.

In [60]:
sentim_analyzer.add_feat_extractor(extract_unigram_feats, unigrams=unigram_feats)

In [61]:
training_set = sentim_analyzer.apply_features(train_docs)
test_set = sentim_analyzer.apply_features(test_docs)

### Treinamento do classificador Naive Bayes

O `SentimentAnalyzer.train` recebe a função de treinamento do `NaiveBayesClassifier` e o corpus transformado. O Naive Bayes aprende a probabilidade condicional $P(\text{feature} \mid \text{classe})$ para cada unigrama e usa o teorema de Bayes para classificar novos documentos, assumindo independência entre as features.

In [62]:
trainer = NaiveBayesClassifier.train
classifier = sentim_analyzer.train(trainer, training_set)

Training classifier


### Features mais informativas

O método `most_informative_features` mostra os tokens com a maior razão de verossimilhança entre as classes. Um valor alto significa que aquela palavra é muito mais provável em uma classe do que na outra — esses são os tokens que o modelo aprendeu a associar com *subjetividade* ou *objetividade*.

In [63]:
for key, value in sorted(classifier.most_informative_features()[:10]):
    print(f'{key}: {value}')

contains(begins): True
contains(both): True
contains(her): True
contains(him): True
contains(his): True
contains(if): True
contains(it): True
contains(its): True
contains(life): True
contains(more): True


### Avaliação no conjunto de teste

O `SentimentAnalyzer.evaluate` calcula as métricas de classificação sobre o conjunto de teste: acurácia, precisão, recall e F-measure para cada classe (`subj` e `obj`).

In [64]:
results_subj = sentim_analyzer.evaluate(test_set)
for key, value in results_subj.items():
    print(f'{key}: {value}')

Evaluating NaiveBayesClassifier results...
Accuracy: 0.8
Precision [obj]: 0.8
Recall [obj]: 0.8
F-measure [obj]: 0.8
Precision [subj]: 0.8
Recall [subj]: 0.8
F-measure [subj]: 0.8


---

## Parte 2: Dataset IMDB — Análise de reviews de filmes

Com o classificador de subjetividade validado, agora aplicamos a mesma abordagem a um problema de sentimento binário real: classificar reviews de filmes do [dataset IMDB](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews) como **positive** ou **negative**.

O dataset contém 50.000 reviews rotuladas. Aqui usamos apenas **1.000** (500 positivas + 500 negativas usadas para teste) para manter o tempo de treinamento razoável.

### Carregando os dados

O dataset é baixado automaticamente via [Kaggle API](https://www.kaggle.com/docs/api) na primeira execução. Para isso é necessário:

1. Criar uma conta no [Kaggle](https://www.kaggle.com) e gerar um token em **Settings → API → Create New Token** — isso baixa um arquivo `kaggle.json`.
2. Colocar o arquivo em `~/.kaggle/kaggle.json` (Linux/Mac) ou `C:\Users\<usuário>\.kaggle\kaggle.json` (Windows).
3. Na primeira execução, a célula abaixo baixa e descompacta o CSV em `data/`. Nas execuções seguintes o arquivo já existe e o download é ignorado.

In [67]:
import os, re
from pathlib import Path

data_path = Path('data/IMDB Dataset.csv')

if not data_path.exists():
    # Lê o token do kaggle.json (suporta formato JSON ou 'export KAGGLE_API_TOKEN=...')
    kaggle_file = Path.home() / '.kaggle' / 'kaggle.json'
    content = kaggle_file.read_text()
    match = re.search(r'KAGGLE_API_TOKEN[=\s:\"\']+([A-Za-z0-9_]+)', content)
    if match:
        os.environ['KAGGLE_API_TOKEN'] = match.group(1)

    data_path.parent.mkdir(exist_ok=True)
    import kaggle
    kaggle.api.dataset_download_files(
        'lakshmi25npathi/imdb-dataset-of-50k-movie-reviews',
        path='data',
        unzip=True,
    )
    print('Download concluído.')
else:
    print('Dataset já presente, pulando download.')

df = pd.read_csv(data_path)
imdb_docs = [(review.split(), sentiment) for review, sentiment in zip(df['review'], df['sentiment'])]

Dataset já presente, pulando download.


### Divisão treino / teste e preparação do corpus

Seguindo o mesmo padrão da Parte 1, separamos **80 %** dos documentos para treino e **20 %** para teste, mantendo o balanceamento entre classes positiva e negativa.

> A tokenização é feita por `str.split()` — simples e sem remoção de stopwords ou pontuação. Isso é intencional para reproduzir o comportamento do exemplo original do NLTK.

In [ ]:
n = 1000

pos_docs = [(words, label) for words, label in imdb_docs if label == 'positive'][:n]
neg_docs = [(words, label) for words, label in imdb_docs if label == 'negative'][:n]

train_docs = pos_docs[:800] + neg_docs[:800]
test_docs = pos_docs[800:] + neg_docs[800:]

### Treinamento do classificador Naive Bayes

O pipeline de treinamento usa o `SentimentAnalyzer` do NLTK com três passos:

1. **Marcação de negação** (`mark_negation`) — palavras após um negador (ex.: *not*, *never*) recebem o sufixo `_NEG` (ex.: `good_NEG`), capturando a inversão de sentimento.
2. **Extração de unigramas** — apenas palavras com frequência ≥ 10 no corpus de treino são usadas como features. Isso remove ruído causado por tokens raros.
3. **Naive Bayes** — classifica assumindo independência condicional entre as features dada a classe.

In [ ]:
sentim_analyzer = SentimentAnalyzer()
all_words_neg = sentim_analyzer.all_words([mark_negation(doc) for doc in train_docs])

unigram_feats = sentim_analyzer.unigram_word_feats(all_words_neg, min_freq=10)
sentim_analyzer.add_feat_extractor(extract_unigram_feats, unigrams=unigram_feats)

training_set = sentim_analyzer.apply_features(train_docs)
test_set = sentim_analyzer.apply_features(test_docs)

trainer = NaiveBayesClassifier.train
classifier = sentim_analyzer.train(trainer, training_set)

### Features mais informativas

O método `most_informative_features` mostra quais tokens têm a razão de verossimilhanças mais extrema entre as classes. Um valor alto indica que a presença daquela feature é muito mais provável em uma classe do que na outra — esses tokens são os "palavras-chave" que o classificador aprendeu a associar com sentimento positivo ou negativo.

In [ ]:
for key, value in sorted(classifier.most_informative_features()[:10]):
    print(f'{key}: {value}')

### Avaliação no conjunto de teste

O `SentimentAnalyzer.evaluate` calcula diversas métricas sobre o conjunto de teste:

| Métrica | O que mede |
|---|---|
| `Accuracy` | Proporção de exemplos classificados corretamente |
| `Precision [positive]` | Dos classificados como positivos, quantos realmente são |
| `Recall [positive]` | Dos positivos reais, quantos foram encontrados |
| `F-measure [positive]` | Média harmônica de Precision e Recall para a classe positiva |
| `Precision [negative]` | Equivalente para a classe negativa |
| `Recall [negative]` | Equivalente para a classe negativa |
| `F-measure [negative]` | Equivalente para a classe negativa |

In [ ]:
results = sentim_analyzer.evaluate(test_set)
for key, value in results.items():
    print(f'{key}: {value}')

### Exemplos individuais classificados pelo Naive Bayes

Para tornar o resultado mais concreto, a tabela abaixo mostra **10 reviews** do conjunto de teste com o rótulo real, a predição do Naive Bayes e um trecho do texto. Os símbolos ✓ e ✗ indicam se o classificador acertou ou não.

In [ ]:
import textwrap
import random

random.seed(42)
# Pega 5 reviews positivas e 5 negativas aleatórias do conjunto de teste
pos_test = [(i, words, label) for i, (words, label) in enumerate(test_docs) if label == 'positive']
neg_test = [(i, words, label) for i, (words, label) in enumerate(test_docs) if label == 'negative']
sample_items = random.sample(pos_test, 5) + random.sample(neg_test, 5)

print(f"{'#':<3} {'Rótulo real':<12} {'NB pred':<12} {'OK?':<5}  Trecho da review")
print("─" * 105)
for i, (idx, words, true_label) in enumerate(sample_items):
    feats, _ = test_set[idx]
    pred = classifier.classify(feats)
    ok = "✓" if pred == true_label else "✗"
    snippet = textwrap.shorten(' '.join(words), width=65, placeholder="…")
    print(f"{i:<3} {true_label:<12} {pred:<12} {ok:<5}  {snippet}")

## Análise com VADER (Valence Aware Dictionary and sEntiment Reasoner)

O VADER é um analisador de sentimento baseado em léxico e regras, incluído no próprio NLTK (`nltk.sentiment.vader`). Ao contrário do classificador Naive Bayes treinado acima, o VADER **não precisa de treinamento** — ele usa um dicionário de polaridade pré-construído e um conjunto de regras heurísticas que lidam com:

- Pontuação (ex.: `!`)
- Capitalização (ex.: `GREAT`)
- Modificadores de intensidade (ex.: *very*, *kind of*)
- Negações (ex.: *not good*)
- Emojis e gírias da internet

O resultado é um dicionário com quatro scores:
- `neg`, `neu`, `pos` — proporção de tokens com sentimento negativo, neutro e positivo
- `compound` — score normalizado entre −1 (mais negativo) e +1 (mais positivo)

In [ ]:
import nltk
nltk.download('vader_lexicon', quiet=True)

from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

### Exemplos do próprio VADER

As frases abaixo são retiradas dos exemplos canônicos do repositório original do VADER e demonstram como o analisador lida com diferentes construções linguísticas.

In [ ]:
# Frases canônicas dos exemplos do VADER (Hutto & Gilbert, 2014)
vader_sentences = [
    # Sentimento positivo básico
    "VADER is smart, handsome, and funny.",
    # Pontuação aumenta intensidade
    "VADER is smart, handsome, and funny!",
    # Capitalização aumenta intensidade
    "VADER is VERY SMART, handsome, and FUNNY.",
    # Modificador de grau
    "VADER is not smart, handsome, nor funny.",
    # Negação inverte o sentimento
    "The book was good.",
    "At least it isn't a horrible book.",
    # "kind of" suaviza o positivo
    "The book was kind of good.",
    # Emojis
    "Make sure you :) or :D today!",
    # Gírias / linguagem informal
    "Today SUX!",
    "Today only kinda sux! But I'll get by, lol",
    # Sentimento misto
    "Most automated sentiment analysis tools are shit.",
    "With a negation: not a good book.",
    # Muito positivo / muito negativo
    "Sentiment analysis has never been THIS easy before!",
    "The plot was good, but the characters are uncompelling and the dialog is not great.",
]

print(f"{'Frase':<65} | neg   | neu   | pos   | compound")
print("-" * 100)
for sentence in vader_sentences:
    scores = sia.polarity_scores(sentence)
    label = f"{scores['neg']:.3f} | {scores['neu']:.3f} | {scores['pos']:.3f} | {scores['compound']:+.4f}"
    print(f"{sentence[:64]:<65} | {label}")

### Classificação por limiar do score `compound`

A recomendação dos autores do VADER é:
- `compound >= 0.05` → **positivo**
- `compound <= -0.05` → **negativo**
- caso contrário → **neutro**

A célula abaixo aplica esse critério e compara com os rótulos do dataset IMDB.

In [ ]:
def vader_label(text):
    compound = sia.polarity_scores(text)['compound']
    if compound >= 0.05:
        return 'positive'
    elif compound <= -0.05:
        return 'negative'
    else:
        return 'neutral'

# Aplica sobre uma amostra do IMDB (1000 reviews para não demorar muito)
sample = df.sample(n=1000, random_state=42).copy()
sample['vader_label'] = sample['review'].apply(vader_label)

# Considera apenas as reviews classificadas como pos/neg (ignora neutral)
classified = sample[sample['vader_label'] != 'neutral']
accuracy = (classified['vader_label'] == classified['sentiment']).mean()

print(f"Reviews classificadas (não-neutras): {len(classified)} de {len(sample)}")
print(f"Acurácia do VADER no subconjunto classificado: {accuracy:.2%}")
print()
print("Distribuição das predições VADER:")
print(sample['vader_label'].value_counts())
print()
print("Distribuição real no sample:")
print(sample['sentiment'].value_counts())

### Exemplos individuais do IMDB classificados pelo VADER

Para facilitar a comparação com o Naive Bayes acima, a tabela abaixo mostra **as mesmas 10 reviews** (5 positivas e 5 negativas) classificadas agora pelo VADER. O coluna `compound` mostra o score contínuo — valores próximos de +1 são fortemente positivos e próximos de −1 são fortemente negativos.

In [ ]:
print(f"{'#':<3} {'Rótulo real':<12} {'VADER pred':<12} {'compound':>9}  {'OK?':<5}  Trecho da review")
print("─" * 105)
for i, (idx, words, true_label) in enumerate(sample_items):
    review_text = ' '.join(words)
    scores = sia.polarity_scores(review_text)
    compound = scores['compound']
    pred = 'positive' if compound >= 0.05 else ('negative' if compound <= -0.05 else 'neutral')
    ok = "✓" if pred == true_label else "✗"
    snippet = textwrap.shorten(review_text, width=65, placeholder="…")
    print(f"{i:<3} {true_label:<12} {pred:<12} {compound:>+9.4f}  {ok:<5}  {snippet}")

---

## Comparação: Naive Bayes vs VADER no dataset IMDB

| Modelo | Treinamento necessário? | Funciona com textos curtos? | Lida com emojis/gírias? |
|---|:---:|:---:|:---:|
| **Naive Bayes (NLTK)** | Sim — requer corpus rotulado | Sim | Não nativamente |
| **VADER** | Não — léxico pré-construído | Sim (focado nisso) | Sim |

A célula abaixo calcula a acurácia de ambos sobre o **mesmo conjunto de teste** (400 reviews IMDB) para uma comparação direta.

In [ ]:
def vader_predict(words):
    c = sia.polarity_scores(' '.join(words))['compound']
    return 'positive' if c >= 0.05 else ('negative' if c <= -0.05 else 'neutral')

# Acurácia sobre o conjunto de teste completo (400 reviews)
nb_correct  = sum(1 for feats, true_label in test_set if classifier.classify(feats) == true_label)
vader_correct = sum(1 for words, true_label in test_docs if vader_predict(words) == true_label)
n_test = len(test_docs)

# Acurácia nos 10 exemplos individuais mostrados acima
nb_10    = sum(1 for _, (idx, words, true_label) in enumerate(sample_items)
               if classifier.classify(test_set[idx][0]) == true_label)
vader_10 = sum(1 for _, (idx, words, true_label) in enumerate(sample_items)
               if vader_predict(words) == true_label)

print(f"Conjunto de teste: {n_test} reviews\n")
print(f"{'Modelo':<22} {'Acurácia geral':>14}  {'Acertos (10 ex.)':>16}")
print("─" * 58)
print(f"{'Naive Bayes':<22} {nb_correct/n_test:>14.2%}  {nb_10:>14}/10")
print(f"{'VADER':<22} {vader_correct/n_test:>14.2%}  {vader_10:>14}/10")
print()
print("Naive Bayes: treinado em 1.600 reviews do próprio IMDB (unigramas ≥ 10×).")
print("VADER: sem treinamento, léxico pré-construído (Hutto & Gilbert, 2014).")